In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform

# ==============================================================================
# CONFIGURARE DATE (Grupele Sanguine)
# ==============================================================================
grupe = ['O', 'A', 'B', 'AB']
probabilitati = [0.46, 0.40, 0.10, 0.04]

# Calculăm probabilitățile cumulative (CDF)
# CDF = [0.46, 0.86, 0.96, 1.00]
cdf = np.cumsum(probabilitati)

def genereaza_grupe_sanguine(N):
    """
    Simulează N persoane folosind Metoda Inversei pentru discret.
    """
    # 1. Generăm N numere uniforme U ~ [0, 1]
    u_values = uniform.rvs(loc=0, scale=1, size=N)

    # 2. Transformăm U în Grupe Sanguine
    # Algoritm: Dacă U < 0.46 -> Grupa O
    #           Dacă 0.46 <= U < 0.86 -> Grupa A, etc.
    rezultate = []
    for u in u_values:
        # np.searchsorted găsește indexul unde s-ar insera u în cdf
        # Este echivalent cu șirul de if/else de mai sus
        idx = np.searchsorted(cdf, u)
        rezultate.append(grupe[idx])

    return rezultate

# ==============================================================================
# VIZUALIZARE ȘI COMPARARE
# ==============================================================================
N_sim = 1000
date_generate = genereaza_grupe_sanguine(N_sim)

# Calculăm frecvențele relative observate
frecvente = {g: date_generate.count(g) / N_sim for g in grupe}
valori_observate = [frecvente[g] for g in grupe]

print("Probabilități Simulate:", frecvente)
print("Probabilități Teoretice:", dict(zip(grupe, probabilitati)))

# Grafic: Bare Simulate vs Teoretice
plt.figure(figsize=(8, 5))
# Bare albastre pentru simulare
plt.bar(grupe, valori_observate, width=0.4, label='Simulare',
        color='skyblue', edgecolor='black', align='center')
# Bare roșii (mai subțiri) pentru teorie
plt.bar(grupe, probabilitati, width=0.2, label='Teorie',
        color='red', alpha=0.7, align='center')

plt.title(f"Distribuția Grupelor Sanguine (N={N_sim})")
plt.ylabel("Frecvență Relativă")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
from scipy.stats import expon
# Atenție: În numpy log este logaritm natural (ln)

# ==============================================================================
# CONFIGURARE (Timp de printare)
# ==============================================================================
# Parametrul lambda (rata) = 1/12
lam = 1/12

def genereaza_exponential(N, lam):
    """
    Generează N valori din Exp(lam) folosind formula inversă.
    """
    # 1. Generăm U ~ Unif(0, 1)
    U = uniform.rvs(loc=0, scale=1, size=N)

    # 2. Aplicăm formula inversă: X = - (1/lambda) * ln(1 - U)
    # Deoarece 1-U este tot uniform pe [0,1], putem folosi simplu ln(U)
    # pentru eficiență, dar formula clasică e cu 1-U.
    X = - (1/lam) * np.log(1 - U)

    return X

# ==============================================================================
# VIZUALIZARE ȘI ESTIMARE
# ==============================================================================
N_sim = 5000
timpi_printare = genereaza_exponential(N_sim, lam)

# i) Histograma și Densitatea Teoretică
plt.figure(figsize=(10, 6))

# Histograma (Simulare)
plt.hist(timpi_printare, bins=30, density=True, color='lightgreen',
         edgecolor='black', alpha=0.6, label='Simulare (Inverse Transform)')

# Densitatea Teoretică (PDF)
# expon.pdf(x, scale=1/lambda) -> scale este media (1/lambda)
x_vals = np.linspace(0, max(timpi_printare), 1000)
pdf_teoretic = expon.pdf(x_vals, scale=1/lam)
plt.plot(x_vals, pdf_teoretic, 'r-', linewidth=2, label='Teorie (PDF)')

plt.title(f"Distribuția Timpului de Printare (Exp, media={1/lam})")
plt.xlabel("Timp (secunde)")
plt.ylabel("Densitate")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ii) Estimare P(T < 5)
# Simulat: Numărăm câte valori sunt < 5
p_simulat = np.mean(timpi_printare < 5)

# Teoretic: CDF(5) = 1 - exp(-lambda * 5)
# Sau folosind scipy: expon.cdf(5, scale=1/lam)
p_teoretic = expon.cdf(5, scale=1/lam)

print(f"Probabilitatea ca timpul să fie sub 5 secunde:")
print(f"   Estimată: {p_simulat:.4f}")
print(f"   Teoretică: {p_teoretic:.4f}")